@app.route("/cache-demo")
def cache_demo():
    resp = make_response({
        "message": "cache demo",
        "timestamp": time.time()
    })
    resp.headers["Cache-Control"] = "public, max-age=60"
    return resp
--------------------------------

manojcloudvm@instance-20260404-024439:~/infra-lab/app1$ curl -i http://127.0.0.1/cache-demo
HTTP/1.1 200 OK
Server: nginx/1.22.1
Date: Tue, 07 Apr 2026 03:21:48 GMT
Content-Type: application/json
Content-Length: 64
Connection: keep-alive
Cache-Control: public, max-age=60

{
  "message": "cache demo",
  "timestamp": 1775532108.116238
}

---
manojcloudvm@instance-20260404-024439:~/infra-lab/app1$ curl -i http://127.0.0.1/cache-demo
HTTP/1.1 200 OK
Server: nginx/1.22.1
Date: Tue, 07 Apr 2026 03:21:53 GMT
Content-Type: application/json
Content-Length: 65
Connection: keep-alive
Cache-Control: public, max-age=60

{
  "message": "cache demo",
  "timestamp": 1775532113.9365487
}

---
manojcloudvm@instance-20260404-024439:~/infra-lab/app1$ curl -i http://127.0.0.1/cache-demo
HTTP/1.1 200 OK
Server: nginx/1.22.1
Date: Tue, 07 Apr 2026 03:21:55 GMT
Content-Type: application/json
Content-Length: 64
Connection: keep-alive
Cache-Control: public, max-age=60

{
  "message": "cache demo",
  "timestamp": 1775532115.129529
}


The difference is just that the second request hit the app again 8 seconds later, so it generated a new timestamp.

What stayed the same
same route: /cache-demo
same cache header: Cache-Control: public, max-age=60
What changed
Date header changed from 03:39:27 to 03:39:35
timestamp in JSON changed too

That means the response was not served from a cache for your curl command.
It was freshly generated both times.

Why

Because the server is only saying:

“this response may be cached for 60 seconds”

It is not saying:
“I will personally return the exact same cached copy myself.”

For caching to actually happen, some client or intermediary must honor that header, such as:

a browser cache
a proxy/cache layer
a CDN

Plain curl here is just making a fresh request each time.

Simple mental model
Cache-Control = caching instructions
actual repeated identical response = only happens if some cache obeys those instructions

So in your example:

cache allowed
cache not used